# Phase 2 — Legal NER: fine-tuning the Phase-1 Efficient-GQA legal LM

Loads `best.pt` + the BPE tokenizer from your Kaggle input datasets (exactly as `capstone-m4` does),
downloads Phase 2 data from Hugging Face with the `HF` Kaggle secret, fine-tunes a BIO token tagger
(optionally with a CRF) over sliding windows that cover the **whole** document, decodes to exact
character spans and writes `submission.zip` (`prediction.json` at the zip root).

**Run order:** top to bottom. Change the switches in section 2 to produce a different submission
(`METHOD` = `full` / `partial` / `lora` / `lorma`; `USE_CRF`; `BIDIRECTIONAL`). Each combination writes to its own
folder under `/kaggle/working/`.

Set `SMOKE = True` in section 2 for a 5-minute end-to-end dry run before a real run.

## 1. Login (Kaggle secret `HF`)

In [1]:
import os, sys, json, math, random, time, zipfile, glob, re, bisect
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in.


## 2. Configuration

In [2]:
# ----------------------------- experiment switches -----------------------------
METHOD         = "lora"   # "full" | "partial" | "lora" | "lorma"
USE_CRF        = True     # linear-chain CRF over BIO tags (False -> class-weighted cross-entropy + constrained Viterbi)
BIDIRECTIONAL  = True     # tokens see both sides while tagging (the LM was pre-trained causal). False = keep causal masks
BIDIR_WINDOW   = 256      # half-width of the sliding-window layers when BIDIRECTIONAL (pre-training used 256 to the left)
PARTIAL_LAYERS = 6        # METHOD == "partial": train only the last N transformer blocks (+ head)
RANK, ALPHA    = 8, 16.0  # METHOD == "lora" / "lorma"

# ----------------------------- data / windows -----------------------------------
WIDTH, STRIDE  = 512, 256 # tokens per window / step between windows (WIDTH must be <= 1024, the pre-training context)
DEV_COUNT      = 8        # documents of fold2 held out for model selection
INFER_BATCH    = 16

# ----------------------------- optimisation (untuned starting points) ------------
BATCH          = 8
EPOCHS         = 50
PATIENCE       = 10
LR             = None     # backbone LR. None -> per-method default below
HEAD_LR        = 5e-4     # tag head + CRF
WEIGHT_DECAY   = 0.01
WARMUP_FRAC    = 0.06
WEIGHT_CAP     = 15.0     # only used when USE_CRF = False
USE_GC         = False    # gradient checkpointing; set True if you hit CUDA OOM
SEED           = 42
SMOKE          = False    # tiny dry run: 4 train docs, 2 dev docs, 1 epoch

# ----------------------------- inputs (None = auto-discover under /kaggle/input) --
CKPT_PATH = None          # e.g. "/kaggle/input/capstone-m4-ckpts/best.pt"
TOK_DIR   = None          # folder holding tokenizer.json + tokenizer_config.json

# # ----------------------------- architecture of the Phase-1 model (from capstone-m4) --
# D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 768, 16, 12, 2
# HEAD_DIM, D_FF, MAX_SEQ_LEN = D_MODEL // N_HEADS, 2048, 1024
# ROPE_BASE, WINDOW_SIZE = 10000.0, 256

# ----------------------------- architecture of the Phase-1 model (from capstone-m4) --
D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 1024, 20, 16, 4
HEAD_DIM, D_FF, MAX_SEQ_LEN = D_MODEL // N_HEADS, 2432, 1024
ROPE_BASE, WINDOW_SIZE = 10000.0, 512

LR_DEFAULTS = {"full": 3e-5, "partial": 5e-5, "lora": 3e-4, "lorma": 3e-4}
assert METHOD in LR_DEFAULTS, METHOD
LR = LR if LR is not None else LR_DEFAULTS[METHOD]
assert WIDTH <= MAX_SEQ_LEN
OUT_DIR = f"/kaggle/working/lner_{METHOD}_{'crf' if USE_CRF else 'ce'}_{'bi' if BIDIRECTIONAL else 'causal'}"
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
print(f"METHOD={METHOD} CRF={USE_CRF} BIDIR={BIDIRECTIONAL} LR={LR} HEAD_LR={HEAD_LR} out={OUT_DIR}")

device: cuda | Tesla T4
METHOD=lora CRF=True BIDIR=True LR=0.0003 HEAD_LR=0.0005 out=/kaggle/working/lner_lora_crf_bi


## 3. Locate `best.pt` and the tokenizer, load the tokenizer

In [3]:
from transformers import PreTrainedTokenizerFast


def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits


if CKPT_PATH is None:
    CKPT_PATH, hits = _find(["/kaggle/input/**/best.pt"], prefer=("m4", "capstone"))
    print("best.pt candidates:", hits)
if TOK_DIR is None:
    tj, hits = _find(["/kaggle/input/**/tokenizer.json"], prefer=("legal_tokenizer", "tokenizer"))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print("tokenizer.json candidates:", hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "best.pt not found - attach the checkpoint dataset or set CKPT_PATH"
assert TOK_DIR and os.path.isdir(TOK_DIR), "tokenizer not found - attach the tokenizer dataset or set TOK_DIR"
print("using checkpoint:", CKPT_PATH)
print("using tokenizer :", TOK_DIR)

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10 ** 9          # we window ourselves; silence the length warning
VOCAB_SIZE = len(tokenizer)
print(f"VOCAB_SIZE={VOCAB_SIZE} EOS={tokenizer.eos_token_id} PAD={tokenizer.pad_token_id} fast={tokenizer.is_fast}")
assert tokenizer.is_fast, "need a fast tokenizer for character offsets"

best.pt candidates: ['/kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt']
tokenizer.json candidates: ['/kaggle/input/datasets/paveetsaluja/m4-dataset-500m/legal_tokenizer_efficient/tokenizer.json']
using checkpoint: /kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt
using tokenizer : /kaggle/input/datasets/paveetsaluja/m4-dataset-500m/legal_tokenizer_efficient
VOCAB_SIZE=32000 EOS=1 PAD=0 fast=True


## 4. Download Phase 2 data from Hugging Face

In [4]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P2 = "Phase_2_LNER"
LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABEL_SET = set(LABELS)
TAG_NAMES = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
K = len(TAG_NAMES)


def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


def load_fold(text_rel, gold_rel=None):
    texts = read_json(fetch(text_rel))
    golds = read_json(fetch(gold_rel)) if gold_rel else {}
    rows, total, dropped, mismatch = [], 0, 0, 0
    for doc_id, text in texts.items():
        ents = []
        for it in golds.get(doc_id, []):
            s, e, lab = int(it[0]), int(it[1]), str(it[2])
            total += 1
            if lab in LABEL_SET and 0 <= s < e <= len(text):
                ents.append((s, e, lab))
                if len(it) > 3 and text[s:e] != it[3]:
                    mismatch += 1
            else:
                dropped += 1
        rows.append((str(doc_id), text, sorted(ents)))
    if gold_rel:
        print(f"{gold_rel}: {len(rows)} docs, {total} entities, dropped {dropped}, text[s:e] != entity_text for {mismatch}")
    return rows


fold1 = load_fold(f"{P2}/Train/fold1-text.json", f"{P2}/Train/fold1-gold.json")
fold2 = load_fold(f"{P2}/Train/fold2-text.json", f"{P2}/Train/fold2-gold.json")
test_rows = load_fold(f"{P2}/Test/fold3-text.json")
print("fold1", len(fold1), "fold2", len(fold2), "test", len(test_rows))

try:
    print("labels.txt:", open(fetch(f"{P2}/Train/labels.txt"), encoding="utf-8").read().split())
except Exception as exc:
    print("could not read labels.txt:", exc)
cnt = Counter(l for _, _, ents in fold1 + fold2 for _, _, l in ents)
print("entities per label:", {l: cnt[l] for l in LABELS})

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold1-gold.json: 35 docs, 4120 entities, dropped 0, text[s:e] != entity_text for 0


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold2-gold.json: 35 docs, 4273 entities, dropped 0, text[s:e] != entity_text for 0


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

fold1 35 fold2 35 test 35


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

labels.txt: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
entities per label: {'APP': 407, 'RESP': 381, 'A.COUNSEL': 195, 'R.COUNSEL': 184, 'JUDGE': 245, 'WIT': 187, 'AUTH': 1003, 'COURT': 1013, 'STAT': 1260, 'PREC': 1183, 'DATE': 1575, 'CASENO': 760}


## 5. Alignment, decoding and scoring utilities
Everything that turns tokens into exact character spans lives here (pure Python/NumPy), and is checked in the next section.

In [5]:
def encode_doc(text):
    """Token ids + character offsets (end-exclusive) with surrounding whitespace removed from every token."""
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offs = []
    for s, e in enc["offset_mapping"]:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        offs.append((s, e))
    return enc["input_ids"], offs


def bio_tags(offs, ents):
    """Tag id per token. A token joins the entity it overlaps most (ties: first entity). Tag ids: 0=O, 1+2i=B-label_i, 2+2i=I-label_i.
    A zero-length (whitespace-only) token strictly inside an entity joins that entity so spans stay contiguous."""
    n = len(offs)
    starts = [o[0] for o in offs]
    ends = [o[1] for o in offs]
    owner = [-1] * n
    best = [0.0] * n
    for ei, (s, e, _lab) in enumerate(ents):
        i = bisect.bisect_right(ends, s)                    # first token whose end > s
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                ov = min(ends[i], e) - max(starts[i], s)
            else:
                ov = 0.5 if (s < starts[i] < e) else 0.0
            if ov > best[i]:
                best[i], owner[i] = ov, ei
            i += 1
    tags = [0] * n
    for i, ei in enumerate(owner):
        if ei < 0:
            continue
        li = LABELS.index(ents[ei][2])
        tags[i] = 2 + 2 * li if (i > 0 and owner[i - 1] == ei) else 1 + 2 * li
    return tags


def window_ranges(n, width, stride):
    if n == 0:
        return []
    out, a = [], 0
    while True:
        b = min(a + width, n)
        out.append((a, b))
        if b == n:
            return out
        a += stride


def trim_span(text, s, e, mode):
    """mode 0: keep token boundaries. 1: drop trailing , ; : and spaces. 2: also drop a trailing '.' that is not an abbreviation dot."""
    if mode == 0:
        return s, e
    while e > s and (text[e - 1] in ",;:" or text[e - 1].isspace()):
        e -= 1
    while s < e and text[s].isspace():
        s += 1
    if mode == 2 and e - s >= 2 and text[e - 1] == "." and text[e - 2].isalnum():
        if not text[s:e].endswith((" v.", " vs.", " No.", " Ltd.", " Pvt.", " Dr.", " Mr.", " Ms.", " Mrs.", " Co.", " Inc.")):
            e -= 1
    return s, e


def decode_spans(text, offs, path, trim_mode=0):
    """BIO tag-id path -> unique (start, end, label) character spans (end exclusive), in document order."""
    spans, seen, i, n = [], set(), 0, len(path)
    while i < n:
        t = int(path[i])
        if t == 0:
            i += 1
            continue
        li = (t - 1) // 2
        j = i + 1
        while j < n and int(path[j]) == 2 + 2 * li:
            j += 1
        nz = [k for k in range(i, j) if offs[k][1] > offs[k][0]]
        if nz:
            s, e = trim_span(text, offs[nz[0]][0], offs[nz[-1]][1], trim_mode)
            sp = (s, e, LABELS[li])
            if e > s and sp not in seen:
                seen.add(sp)
                spans.append(sp)
        i = j
    return spans


def allowed_matrix(tag_names):
    k = len(tag_names)
    allowed = np.zeros((k, k), dtype=bool)
    for a, sa in enumerate(tag_names):
        for b, sb in enumerate(tag_names):
            if sb == "O" or sb.startswith("B-"):
                allowed[a, b] = True
            else:
                allowed[a, b] = sa in ("B-" + sb[2:], "I-" + sb[2:])
    return allowed


ALLOWED = allowed_matrix(TAG_NAMES)
HARD_TRANS = np.where(ALLOWED, 0.0, -1e4)          # transition scores used when there is no CRF


def log_softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=-1, keepdims=True))


def viterbi(em, trans):
    """em: (T,K) emission scores, trans: (K,K) transition scores -> best tag-id path (list)."""
    T, k = em.shape
    if T == 0:
        return []
    score = em[0].copy()
    back = np.zeros((T, k), dtype=np.int32)
    cols = np.arange(k)
    for t in range(1, T):
        cand = score[:, None] + trans
        idx = cand.argmax(axis=0)
        back[t] = idx
        score = cand[idx, cols] + em[t]
    cur = int(score.argmax())
    path = [cur]
    for t in range(T - 1, 0, -1):
        cur = int(back[t, cur])
        path.append(cur)
    return path[::-1]


def best_path(logits, trans, use_crf, o_delta=0.0):
    em = logits.astype(np.float64) if use_crf else log_softmax_np(logits.astype(np.float64))
    em[:, 0] -= o_delta                       # o_delta > 0 trades precision for recall
    return viterbi(em, trans)


def strict_scores(gold_rows, preds):
    """Strict span+label macro P/R/F1 over the 12 labels (counts pooled over documents, as in the reference)."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc_id, _text, ents in gold_rows:
        g = set(ents)
        p = set(tuple(x[:3]) for x in preds.get(doc_id, []))
        for l in LABELS:
            gl = {x for x in g if x[2] == l}
            pl = {x for x in p if x[2] == l}
            tp[l] += len(gl & pl)
            fp[l] += len(pl - gl)
            fn[l] += len(gl - pl)
    per, ps, rs, fs = {}, [], [], []
    for l in LABELS:
        pr = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        rc = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = (pr, rc, f1, tp[l], fp[l], fn[l])
        ps.append(pr); rs.append(rc); fs.append(f1)
    return {"f1": float(np.mean(fs)), "p": float(np.mean(ps)), "r": float(np.mean(rs)), "per": per}


def print_scores(sc):
    for l in LABELS:
        pr, rc, f1, tp_, fp_, fn_ = sc["per"][l]
        print(f"  {l:10s} P={pr:.3f} R={rc:.3f} F1={f1:.3f}  (tp {tp_}, fp {fp_}, fn {fn_})")
    print(f"  MACRO      P={sc['p']:.3f} R={sc['r']:.3f} F1={sc['f1']:.3f}")

## 6. Check the offset pipeline on the real data (before any training)
* **Boundary report**: how many gold spans start and end exactly on token boundaries (the ceiling for a token tagger).
* **Oracle round trip**: gold -> BIO tags -> spans. This must be (close to) 1.0; whatever is lost is lost to tokenization, not to the model.
* Picks the trailing-punctuation rule (`TRIM_MODE`) that makes the oracle best.

In [6]:
PREP = {}                                  # doc_id -> (ids, offsets)


def prepare(rows):
    for doc_id, text, _ in tqdm(rows, desc="tokenizing", leave=False):
        if doc_id not in PREP:
            PREP[doc_id] = encode_doc(text)


all_train = fold1 + fold2
prepare(all_train)
lens = np.array([len(PREP[d][0]) for d, _, _ in all_train])
print(f"tokens/doc: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}; docs longer than WIDTH: {(lens > WIDTH).mean():.0%}")

# whitespace-trimmed offsets must be ordered, inside the text, and never start/end on whitespace
for doc_id, text, _ in all_train[:20]:
    ids, offs = PREP[doc_id]
    assert all(0 <= s <= e <= len(text) for s, e in offs), "offset outside text"
    assert all(a[0] <= b[0] and a[1] <= b[1] for a, b in zip(offs, offs[1:])), "offsets not monotonic"

covered = total = shown = 0
for doc_id, text, ents in all_train:
    ids, offs = PREP[doc_id]
    st = {s for s, e in offs if e > s}
    en = {e for s, e in offs if e > s}
    for s, e, l in ents:
        total += 1
        if s in st and e in en:
            covered += 1
        elif shown < 8:
            shown += 1
            print(f"  off-boundary {l}: {text[s:e][:50]!r} in ...{text[max(0, s - 10):e + 10][:80]!r}")
print(f"gold spans on token boundaries: {covered}/{total} = {covered / max(total, 1):.3f}")

oracle = {}
for mode in (0, 1, 2):
    preds = {d: decode_spans(t, PREP[d][1], bio_tags(PREP[d][1], ents), mode) for d, t, ents in all_train}
    oracle[mode] = strict_scores(all_train, preds)
    print(f"oracle round-trip, trim mode {mode}: macro-F1 {oracle[mode]['f1']:.4f}")
TRIM_MODE = max((0, 1, 2), key=lambda m: (oracle[m]["f1"], -m))
print("TRIM_MODE =", TRIM_MODE, "(oracle per label below)")
print_scores(oracle[TRIM_MODE])

tokenizing:   0%|          | 0/70 [00:00<?, ?it/s]

tokens/doc: mean 6550, median 5538, max 23814; docs longer than WIDTH: 100%
  off-boundary PREC: 'State of M.P. vs. Kashiram & Ors.' in ...'e case of State of M.P. vs. Kashiram & Ors.[1], the s'
  off-boundary PREC: 'Gambangi Appalaswamy Naldu and Others v. Behara Ve' in ...'1565] and Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro'
  off-boundary AUTH: 'Railway Board' in ...' down the Railway Boards instruct'
  off-boundary AUTH: 'Railway Board' in ...'tation of Railway Boards decision'
  off-boundary PREC: 'Shankarsan Dash Vs. Union of India [1991 (3) SCC 4' in ...' cases in Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]. Para 7 o'
  off-boundary PREC: 'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 1' in ...'C 268] or Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]. Hence th'
  off-boundary STAT: 'Constitution' in ...'After the Constitutional Amendme'
  off-boundary AUTH: 'State Government' in ...'uding the State Governments and the '
gold spans 

## 7. Phase-1 model (copied from `capstone-m4`) and checkpoint loading

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight


def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)


class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)

    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents


base = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                        MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]
# capstone-m4 saved the torch.compile()d model, so keys carry an "_orig_mod." prefix
state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in state.items()}
missing, unexpected = base.load_state_dict(state, strict=False)
missing = [k for k in missing if k != "lm_head.weight"]      # tied to tok_emb.weight, so a copy is not required
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
print(f"loaded {CKPT_PATH}  (pre-training step {ckpt.get('step')}, loss {ckpt.get('loss')})")
del ckpt, state
base = base.to(dev).eval()

# sanity: a correctly loaded LM has a low next-token loss on legal text (random weights give ~ln(32000) = 10.4)
ids0 = torch.tensor([PREP[all_train[0][0]][0][:512]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
    lm_loss = base(ids0, targets=ids0)[1].item()
print(f"LM sanity loss on a training judgment: {lm_loss:.3f}  (expect roughly 2-4)")
assert lm_loss < 6.0, "LM loss is too high - weights or tokenizer do not match"

loaded /kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt  (pre-training step 7629, loss 2.381956102848053)
LM sanity loss on a training judgment: 2.307  (expect roughly 2-4)


## 8. Tagger: encoder pass (optionally bidirectional), adapters, CRF

In [8]:
def build_masks(T, pad_mask, bidirectional, window):
    """Boolean attention masks (True = may attend), shape (B,1,T,T): one for window layers, one for full layers."""
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    if bidirectional:
        full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
        win = (c - r).abs() < window
    else:
        full = c <= r
        win = (c <= r) & (c > r - window)
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key


def block_forward(blk, x, cos, sin, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))


class LoRALinear(nn.Module):
    """y = W x + (alpha/r) B A x   (base W frozen, B starts at zero)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative): y = (I + (alpha/r) B A) W x   (base W frozen, B starts at zero -> identical to Phase 1 at step 0)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())


def adapt_linears(backbone, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, wrapper(child, rank, alpha))
                    n += 1
    return n


class ChainCRF(nn.Module):
    """Linear-chain CRF over BIO tags; illegal transitions (O->I-x, B-x->I-y, ...) are blocked."""
    def __init__(self, tag_names):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(len(tag_names), len(tag_names)))
        self.register_buffer("allowed", torch.from_numpy(allowed_matrix(tag_names)))

    def transition(self):
        return self.transitions.masked_fill(~self.allowed, -1e4)

    def nll(self, em, tags, mask):
        trans = self.transition()
        safe = tags.clamp(min=0)
        mf = mask.float()
        emit = em.gather(2, safe.unsqueeze(2)).squeeze(2)
        gold = (emit * mf).sum(1) + (trans[safe[:, :-1], safe[:, 1:]] * mf[:, 1:]).sum(1)
        alpha = em[:, 0]
        for t in range(1, em.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0) + em[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        logz = torch.logsumexp(alpha, dim=-1)
        return ((logz - gold) / mf.sum(1).clamp(min=1)).mean()


class LegalTagger(nn.Module):
    def __init__(self, backbone, tag_names, use_crf, bidirectional, bidir_window, use_gc):
        super().__init__()
        self.backbone = backbone
        self.bidirectional, self.bidir_window, self.use_gc = bidirectional, bidir_window, use_gc
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(D_MODEL, len(tag_names))
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        with torch.no_grad():
            self.head.bias[0] = 2.0                      # start by predicting "O"
        self.crf = ChainCRF(tag_names) if use_crf else None

    def encode(self, ids, pad_mask):
        m = self.backbone
        T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        window = self.bidir_window if self.bidirectional else WINDOW_SIZE
        full, win = build_masks(T, pad_mask, self.bidirectional, window)
        for blk in m.layers:
            mask = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mask, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mask)
        return m.final_norm(x)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.encode(ids, pad_mask)))


def build_tagger():
    bb = base
    for p in bb.parameters():
        p.requires_grad_(METHOD == "full")
    if METHOD == "partial":
        for blk in bb.layers[-PARTIAL_LAYERS:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in bb.final_norm.parameters():
            p.requires_grad_(True)
    elif METHOD in ("lora", "lorma"):
        print(f"wrapped {adapt_linears(bb, METHOD, RANK, ALPHA)} linear layers with {METHOD}")
    tg = LegalTagger(bb, TAG_NAMES, USE_CRF, BIDIRECTIONAL, BIDIR_WINDOW, USE_GC).to(dev)
    tr = sum(p.numel() for p in tg.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in tg.parameters())
    print(f"parameters: trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.1%})")
    return tg

## 9. Windows, split, model, optimiser

In [9]:
names_index = {n: i for i, n in enumerate(TAG_NAMES)}
fold2_sorted = sorted(fold2, key=lambda r: r[0])
dev_rows = fold2_sorted[-DEV_COUNT:]
dev_ids = {r[0] for r in dev_rows}
train_rows = list(fold1) + [r for r in fold2 if r[0] not in dev_ids]
if SMOKE:
    train_rows, dev_rows, EPOCHS = train_rows[:4], dev_rows[:2], 1
print("train docs", len(train_rows), "| dev docs", [r[0] for r in dev_rows])
prepare(train_rows + dev_rows + test_rows)

train_windows = []                                           # (token ids, tag ids)
for doc_id, text, ents in train_rows:
    ids, offs = PREP[doc_id]
    tags = bio_tags(offs, ents)
    for a, b in window_ranges(len(ids), WIDTH, STRIDE):
        train_windows.append((ids[a:b], tags[a:b]))
print(f"train windows: {len(train_windows)} (width {WIDTH}, stride {STRIDE})")

tag_counts = np.ones(K)
for _ids, tgs in train_windows:
    for t in tgs:
        tag_counts[t] += 1
w = 1.0 / np.sqrt(tag_counts)
w = w * (K / w.sum())
w = np.minimum(w, np.median(w) * WEIGHT_CAP)
cls_weights = torch.tensor(w, dtype=torch.float32, device=dev)   # used only when USE_CRF = False


def collate(items):
    L = max(len(i) for i, _ in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    tgt = torch.full((len(items), L), -100, dtype=torch.long)
    pm = torch.zeros(len(items), L, dtype=torch.bool)
    for r, (i, t) in enumerate(items):
        ids[r, :len(i)] = torch.tensor(i)
        tgt[r, :len(t)] = torch.tensor(t)
        pm[r, :len(i)] = True
    return ids.to(dev), tgt.to(dev), pm.to(dev)


tagger = build_tagger()
trainable = [p for p in tagger.parameters() if p.requires_grad]
bb_params = [p for n, p in tagger.named_parameters() if p.requires_grad and n.startswith("backbone.")]
hd_params = [p for n, p in tagger.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
groups = [{"params": hd_params, "lr": HEAD_LR}] + ([{"params": bb_params, "lr": LR}] if bb_params else [])
optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_windows) / BATCH)
total_steps = steps_per_epoch * EPOCHS
warm = max(1, int(WARMUP_FRAC * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: (s + 1) / warm if s < warm else max(0.05, (total_steps - s) / max(1, total_steps - warm)))
use_amp = dev.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"steps/epoch {steps_per_epoch}, total {total_steps}, warmup {warm}")

train docs 62 | dev docs ['756812', '76640285', '79816183', '806212', '83742049', '87753577', '90251163', '952082']


tokenizing:   0%|          | 0/105 [00:00<?, ?it/s]

train windows: 1541 (width 512, stride 256)
wrapped 140 linear layers with lora
parameters: trainable 2.75M / total 237.41M (1.2%)
steps/epoch 193, total 9650, warmup 579


## 10. Inference helpers (whole document, overlapping windows) and dev scoring

In [10]:
@torch.no_grad()
def doc_logits(tg, ids):
    """Per-token tag logits for a whole document: overlapping windows, centre-weighted average. -> (n_tokens, K) float32."""
    tg.eval()
    n = len(ids)
    ids_t = torch.tensor(ids, dtype=torch.long)
    acc = torch.zeros(n, K, device=dev)
    wsum = torch.zeros(n, 1, device=dev)
    ranges = window_ranges(n, WIDTH, STRIDE)
    for i in range(0, len(ranges), INFER_BATCH):
        chunk = ranges[i:i + INFER_BATCH]
        L = max(b - a for a, b in chunk)
        x = torch.zeros(len(chunk), L, dtype=torch.long)
        pm = torch.zeros(len(chunk), L, dtype=torch.bool)
        for r, (a, b) in enumerate(chunk):
            x[r, :b - a] = ids_t[a:b]
            pm[r, :b - a] = True
        x, pm = x.to(dev), pm.to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            out = tg(x, pm)
        out = out.float()
        for r, (a, b) in enumerate(chunk):
            m = b - a
            wt = torch.minimum(torch.arange(1, m + 1, device=dev), torch.arange(m, 0, -1, device=dev)).float().unsqueeze(1)
            acc[a:b] += out[r, :m] * wt
            wsum[a:b] += wt
    return (acc / wsum.clamp(min=1e-6)).cpu().numpy()


def get_trans(tg):
    if tg.crf is not None:
        return tg.crf.transition().detach().float().cpu().numpy().astype(np.float64)
    return HARD_TRANS


def build_cache(tg, rows, desc="scoring"):
    return [(d, t, PREP[d][1], doc_logits(tg, PREP[d][0])) for d, t, _ in tqdm(rows, desc=desc, leave=False)]


def decode_cache(cache, trans, o_delta=0.0, trim_mode=0):
    return {d: decode_spans(t, offs, best_path(lg, trans, USE_CRF, o_delta), trim_mode) for d, t, offs, lg in cache}


def evaluate_dev(tg):
    cache = build_cache(tg, dev_rows)
    return strict_scores(dev_rows, decode_cache(cache, get_trans(tg), 0.0, TRIM_MODE))

## 11. Train (keeps the best epoch by dev strict macro-F1)

In [11]:
best_path_file = os.path.join(OUT_DIR, "best_lner.pt")
best_f1, bad, history = -1.0, 0, []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tagger.train()
    order = np.random.permutation(len(train_windows))
    tot_loss, nb = 0.0, 0
    bar = tqdm(range(0, len(order), BATCH), desc=f"epoch {epoch}/{EPOCHS}", leave=False)
    for bi in bar:
        ids, tgt, pm = collate([train_windows[i] for i in order[bi:bi + BATCH]])
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            logits = tagger(ids, pm)
        logits = logits.float()
        if tagger.crf is not None:
            loss = tagger.crf.nll(logits, tgt, pm)
        else:
            loss = F.cross_entropy(logits.reshape(-1, K), tgt.reshape(-1), weight=cls_weights, ignore_index=-100)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(optimizer)
        scaler.update()
        sched.step()
        tot_loss += float(loss.detach()); nb += 1
        bar.set_postfix(loss=f"{tot_loss / nb:.4f}")
    sc = evaluate_dev(tagger)
    history.append({"epoch": epoch, "train_loss": tot_loss / nb, "dev_f1": sc["f1"], "dev_p": sc["p"], "dev_r": sc["r"]})
    print(f"epoch {epoch}: train loss {tot_loss / nb:.4f} | dev strict macro P {sc['p']:.3f} R {sc['r']:.3f} F1 {sc['f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
    if sc["f1"] > best_f1:
        best_f1, bad = sc["f1"], 0
        torch.save({"model": tagger.state_dict(), "dev_f1": best_f1, "epoch": epoch}, best_path_file)
        print("  saved best")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stop")
            break
print("best dev macro-F1:", round(best_f1, 4))

epoch 1/50:   0%|          | 0/193 [00:00<?, ?it/s]

/tmp/ipykernel_23/1034448291.py:24: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


scoring:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 1: train loss 0.8122 | dev strict macro P 0.303 R 0.152 F1 0.1876 | 2.6 min
  saved best


epoch 2/50:   0%|          | 0/193 [00:00<?, ?it/s]

scoring:   0%|          | 0/8 [00:00<?, ?it/s]

... (36 outputs left out)

epoch 14: train loss 0.0293 | dev strict macro P 0.591 R 0.383 F1 0.4134 | 37.6 min


epoch 15/50:   0%|          | 0/193 [00:00<?, ?it/s]

scoring:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 15: train loss 0.0314 | dev strict macro P 0.461 R 0.329 F1 0.3437 | 40.4 min
early stop
best dev macro-F1: 0.5315


## 12. Reload the best epoch, tune decoding on dev (trailing-punctuation rule x "O" penalty), per-label report

In [12]:
tagger.load_state_dict(torch.load(best_path_file, map_location=dev)["model"])
trans = get_trans(tagger)
dev_cache = build_cache(tagger, dev_rows, "dev logits")

DELTAS = [0.0, -0.5, 0.5, 1.0, 1.5, 2.0, 3.0]
MODES = [TRIM_MODE] + [m for m in (0, 1, 2) if m != TRIM_MODE]
best_cfg, best_sc = (0.0, TRIM_MODE), None
for delta in DELTAS:
    paths = [(d, t, offs, best_path(lg, trans, USE_CRF, delta)) for d, t, offs, lg in dev_cache]
    for mode in MODES:
        sc = strict_scores(dev_rows, {d: decode_spans(t, offs, p, mode) for d, t, offs, p in paths})
        if best_sc is None or sc["f1"] > best_sc["f1"] + 1e-9:
            best_sc, best_cfg = sc, (delta, mode)
        print(f"  o_delta {delta:+.1f} trim {mode}: F1 {sc['f1']:.4f} (P {sc['p']:.3f} R {sc['r']:.3f})")
O_DELTA, FINAL_TRIM = best_cfg
print(f"\nchosen o_delta={O_DELTA}, trim_mode={FINAL_TRIM}  -> dev strict macro-F1 {best_sc['f1']:.4f}")
print_scores(best_sc)

dev logits:   0%|          | 0/8 [00:00<?, ?it/s]

  o_delta +0.0 trim 1: F1 0.5315 (P 0.567 R 0.530)
  o_delta +0.0 trim 0: F1 0.5299 (P 0.566 R 0.528)
  o_delta +0.0 trim 2: F1 0.5262 (P 0.567 R 0.521)
  o_delta -0.5 trim 1: F1 0.5030 (P 0.597 R 0.476)
  o_delta -0.5 trim 0: F1 0.5023 (P 0.596 R 0.475)
  o_delta -0.5 trim 2: F1 0.4977 (P 0.597 R 0.468)
  o_delta +0.5 trim 1: F1 0.5309 (P 0.532 R 0.563)
  o_delta +0.5 trim 0: F1 0.5294 (P 0.531 R 0.561)
  o_delta +0.5 trim 2: F1 0.5253 (P 0.530 R 0.553)
  o_delta +1.0 trim 1: F1 0.5151 (P 0.505 R 0.575)
  o_delta +1.0 trim 0: F1 0.5136 (P 0.504 R 0.572)
  o_delta +1.0 trim 2: F1 0.5093 (P 0.502 R 0.564)
  o_delta +1.5 trim 1: F1 0.4966 (P 0.479 R 0.570)
  o_delta +1.5 trim 0: F1 0.4920 (P 0.476 R 0.564)
  o_delta +1.5 trim 2: F1 0.4905 (P 0.476 R 0.560)
  o_delta +2.0 trim 1: F1 0.4691 (P 0.452 R 0.551)
  o_delta +2.0 trim 0: F1 0.4625 (P 0.447 R 0.543)
  o_delta +2.0 trim 2: F1 0.4617 (P 0.447 R 0.540)
  o_delta +3.0 trim 1: F1 0.3520 (P 0.350 R 0.408)
  o_delta +3.0 trim 0: F1 0.350

## 13. Error analysis on dev (for the report)

In [13]:
dev_preds = decode_cache(dev_cache, trans, O_DELTA, FINAL_TRIM)
kinds = defaultdict(Counter)
for d, text, ents in dev_rows:
    G, P = set(ents), set(dev_preds[d])
    for (s, e, l) in P - G:
        if any((s, e) == (gs, ge) for gs, ge, _ in G):
            kinds[l]["wrong label"] += 1
        elif any(gl == l and s < ge and gs < e for gs, ge, gl in G):
            kinds[l]["boundary error"] += 1
        elif any(s < ge and gs < e for gs, ge, _ in G):
            kinds[l]["overlaps other label"] += 1
        else:
            kinds[l]["spurious"] += 1
    for (s, e, l) in G - P:
        if not any(s < pe and ps < e for ps, pe, _ in P):
            kinds[l]["missed"] += 1
print(f"{'label':10s} {'wrong label':>12s} {'boundary':>9s} {'overlap other':>14s} {'spurious':>9s} {'missed':>7s}")
for l in LABELS:
    c = kinds[l]
    print(f"{l:10s} {c['wrong label']:12d} {c['boundary error']:9d} {c['overlap other label']:14d} {c['spurious']:9d} {c['missed']:7d}")

label       wrong label  boundary  overlap other  spurious  missed
APP                   4         4              0        12      43
RESP                  3         3              0        16      61
A.COUNSEL             1         1              0         0       1
R.COUNSEL            10         0              0         2       4
JUDGE                 1         1              0         5       2
WIT                   0         2              0         4      11
AUTH                 13         5              0        31      42
COURT                 0         4              0         8       6
STAT                  1        27              0        12       7
PREC                 11        19              0        45       5
DATE                  1         2              0         6       1
CASENO                0        14              0         4      27


## 14. Predict the test documents, write `submission.zip`, validate it

In [14]:
test_cache = build_cache(tagger, test_rows, "test logits")
test_preds = decode_cache(test_cache, trans, O_DELTA, FINAL_TRIM)

payload = {doc_id: [[int(s), int(e), lab] for s, e, lab in test_preds[doc_id]] for doc_id, _, _ in test_rows}
pred_path = os.path.join(OUT_DIR, "prediction.json")
zip_path = os.path.join(OUT_DIR, "submission.zip")
with open(pred_path, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, ensure_ascii=False)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")          # must sit at the root of the zip

# ---- validation: re-open what we are about to upload ----
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
    check = json.loads(z.read("prediction.json"))
texts = {d: t for d, t, _ in test_rows}
assert set(check) == set(texts), "ids differ from the official test ids"
n_spans, ws_spans = 0, 0
for d, spans in check.items():
    seen = set()
    for s, e, lab in spans:
        assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(texts[d]), (d, s, e)
        assert lab in LABEL_SET, lab
        assert (s, e, lab) not in seen, ("duplicate span", d, s, e, lab)
        seen.add((s, e, lab))
        ws_spans += texts[d][s:e] != texts[d][s:e].strip()
        n_spans += 1
print(f"OK: {len(check)} documents, {n_spans} spans -> {zip_path} ({os.path.getsize(zip_path) / 1024:.1f} KB)")
if ws_spans:
    print(f"WARNING: {ws_spans} spans start or end on whitespace")
print("label counts:", dict(Counter(l for sp in check.values() for _, _, l in sp)))
d0 = test_rows[0][0]
for s, e, lab in check[d0][:8]:
    print(f"  {d0} [{s}:{e}] {lab:10s} {texts[d0][s:e]!r}")

with open(os.path.join(OUT_DIR, "run_config.json"), "w") as fh:
    json.dump({"METHOD": METHOD, "USE_CRF": USE_CRF, "BIDIRECTIONAL": BIDIRECTIONAL, "LR": LR, "HEAD_LR": HEAD_LR, "WIDTH": WIDTH,
               "STRIDE": STRIDE, "BATCH": BATCH, "EPOCHS_RUN": len(history), "O_DELTA": O_DELTA, "TRIM_MODE": FINAL_TRIM,
               "dev_macro_f1": best_sc["f1"], "history": history}, fh, indent=2)

test logits:   0%|          | 0/35 [00:00<?, ?it/s]

OK: 35 documents, 4019 spans -> /kaggle/working/lner_lora_crf_bi/submission.zip (26.9 KB)
label counts: {'COURT': 549, 'RESP': 73, 'APP': 88, 'JUDGE': 118, 'DATE': 745, 'CASENO': 246, 'AUTH': 431, 'STAT': 791, 'PREC': 736, 'A.COUNSEL': 110, 'R.COUNSEL': 74, 'WIT': 58}
  1128835 [18:40] COURT      'SUPREME COURT OF INDIA'
  1128835 [169:183] RESP       'FABS PVT. LTD.'
  1128835 [213:229] RESP       'STATE OF HARYANA'
  1128835 [355:391] APP        'PANIPAT TEACHERS (RECOGNISED SCHOOLS'
  1128835 [392:424] APP        'HOUSING CO-OPEATIVE SOCIETY LTD.'
  1128835 [454:470] APP        'STATE OF HARYANA'
  1128835 [634:648] RESP       'FABS PVT. LTD.'
  1128835 [679:695] RESP       'STATE OF HARYANA'


In [15]:
# import gc

# # 1. Define the parameters for the loop
# METHODS_TO_TEST = ["full", "partial", "lora", "lorma"]
# EPOCHS = 10  # Enforce 10 epochs for every method

# for current_method in METHODS_TO_TEST:
#     print(f"\n{'='*80}")
#     print(f"🚀 STARTING EXPERIMENT: {current_method.upper()}")
#     print(f"{'='*80}")
    
#     # 2. Update global variables for the current method
#     global METHOD, LR, OUT_DIR, base
#     METHOD = current_method
#     LR = LR_DEFAULTS[METHOD]
#     OUT_DIR = f"/kaggle/working/lner_{METHOD}_{'crf' if USE_CRF else 'ce'}_{'bi' if BIDIRECTIONAL else 'causal'}"
#     os.makedirs(OUT_DIR, exist_ok=True)
    
#     # 3. Reload a FRESH Base Model (Crucial: prevents adapter stacking from previous loops)
#     print("Reloading a fresh Phase-1 base model...")
#     base = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
#                             MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
    
#     # Load weights
#     ckpt_tmp = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
#     state_tmp = ckpt_tmp["model_state_dict"]
#     state_tmp = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in state_tmp.items()}
#     base.load_state_dict(state_tmp, strict=False)
#     base = base.to(dev).eval()
#     del ckpt_tmp, state_tmp
#     gc.collect()
#     torch.cuda.empty_cache()

#     # 4. Build Tagger and Optimizer
#     tagger = build_tagger()
#     trainable = [p for p in tagger.parameters() if p.requires_grad]
#     bb_params = [p for n, p in tagger.named_parameters() if p.requires_grad and n.startswith("backbone.")]
#     hd_params = [p for n, p in tagger.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    
#     groups = [{"params": hd_params, "lr": HEAD_LR}] + ([{"params": bb_params, "lr": LR}] if bb_params else [])
#     optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    
#     steps_per_epoch = math.ceil(len(train_windows) / BATCH)
#     total_steps = steps_per_epoch * EPOCHS
#     warm = max(1, int(WARMUP_FRAC * total_steps))
#     sched = torch.optim.lr_scheduler.LambdaLR(
#         optimizer, lambda s: (s + 1) / warm if s < warm else max(0.05, (total_steps - s) / max(1, total_steps - warm)))
#     scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    
#     # 5. Training Loop
#     best_path_file = os.path.join(OUT_DIR, f"best_lner_{METHOD}.pt")
#     best_f1, bad = -1.0, 0
#     t0 = time.time()
    
#     for epoch in range(1, EPOCHS + 1):
#         tagger.train()
#         order = np.random.permutation(len(train_windows))
#         tot_loss, nb = 0.0, 0
#         bar = tqdm(range(0, len(order), BATCH), desc=f"{METHOD.upper()} Epoch {epoch}/{EPOCHS}", leave=False)
        
#         for bi in bar:
#             ids, tgt, pm = collate([train_windows[i] for i in order[bi:bi + BATCH]])
#             with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
#                 logits = tagger(ids, pm)
#             logits = logits.float()
            
#             if tagger.crf is not None:
#                 loss = tagger.crf.nll(logits, tgt, pm)
#             else:
#                 loss = F.cross_entropy(logits.reshape(-1, K), tgt.reshape(-1), weight=cls_weights, ignore_index=-100)
                
#             optimizer.zero_grad(set_to_none=True)
#             scaler.scale(loss).backward()
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(trainable, 1.0)
#             scaler.step(optimizer)
#             scaler.update()
#             sched.step()
            
#             tot_loss += float(loss.detach())
#             nb += 1
#             bar.set_postfix(loss=f"{tot_loss / nb:.4f}")
            
#         # Evaluate on Dev Split
#         sc = evaluate_dev(tagger)
#         print(f"Epoch {epoch}: Train Loss {tot_loss / nb:.4f} | Dev Strict Macro-F1 {sc['f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
        
#         if sc["f1"] > best_f1:
#             best_f1, bad = sc["f1"], 0
#             torch.save({"model": tagger.state_dict(), "dev_f1": best_f1, "epoch": epoch}, best_path_file)
#             print("  --> Saved new best checkpoint")
#         else:
#             bad += 1
#             if bad >= PATIENCE:
#                 print("  --> Early stop triggered")
#                 break

#     # 6. Tune Thresholds on Best Epoch
#     print(f"\nEvaluating Best {METHOD.upper()} Checkpoint...")
#     tagger.load_state_dict(torch.load(best_path_file, map_location=dev)["model"])
#     trans = get_trans(tagger)
#     dev_cache = build_cache(tagger, dev_rows, f"dev logits ({METHOD})")
    
#     DELTAS = [0.0, -0.5, 0.5, 1.0, 1.5, 2.0, 3.0]
#     MODES = [TRIM_MODE] + [m for m in (0, 1, 2) if m != TRIM_MODE]
#     best_cfg, best_sc = (0.0, TRIM_MODE), None
    
#     for delta in DELTAS:
#         paths = [(d, t, offs, best_path(lg, trans, USE_CRF, delta)) for d, t, offs, lg in dev_cache]
#         for mode in MODES:
#             sc = strict_scores(dev_rows, {d: decode_spans(t, offs, p, mode) for d, t, offs, p in paths})
#             if best_sc is None or sc["f1"] > best_sc["f1"] + 1e-9:
#                 best_sc, best_cfg = sc, (delta, mode)
                
#     O_DELTA, FINAL_TRIM = best_cfg
#     print(f"Chosen Params -> o_delta: {O_DELTA}, trim_mode: {FINAL_TRIM} | Final Dev F1: {best_sc['f1']:.4f}")

#     # 7. Predict Test Set & Package Custom Submission Zip
#     print(f"\nGenerating Test Predictions for {METHOD.upper()}...")
#     test_cache = build_cache(tagger, test_rows, f"test logits ({METHOD})")
#     test_preds = decode_cache(test_cache, trans, O_DELTA, FINAL_TRIM)
    
#     payload = {doc_id: [[int(s), int(e), lab] for s, e, lab in test_preds[doc_id]] for doc_id, _, _ in test_rows}
    
#     # Save isolated prediction.json temporarily
#     pred_path = os.path.join(OUT_DIR, "prediction.json")
#     with open(pred_path, "w", encoding="utf-8") as fh:
#         json.dump(payload, fh, ensure_ascii=False)
        
#     # Create method-specific submission zip in the working root
#     final_zip_path = f"/kaggle/working/submission_{METHOD}.zip"
#     with zipfile.ZipFile(final_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
#         z.write(pred_path, arcname="prediction.json")
        
#     print(f"✅ SUCCESSFULLY SAVED: {final_zip_path}")
    
#     # Free memory for the next loop
#     del tagger, optimizer, scaler, dev_cache, test_cache
#     gc.collect()
#     torch.cuda.empty_cache()

# print("\n🎉 ALL 4 METHODS COMPLETED!")